# Notebook 3: Check a file of your own

Does a series that an operator brings fit what the system expects? This notebook checks a CSV file against a
signal: unit, interval, time zone, value range, gaps, duplicate times. Then it puts the file next to a related series
of the system and says at the end whether and how it could be bound.

The sample is `beispiel-windmessung.csv`: wind in km/h, local time without a time zone, decimal comma, with one
outlier and one gap. For a file of your own, change the settings in the first code cell.

The notebook reads the file and the retrieval server. It writes only a prepared copy of the file into its own
directory; it writes nothing to the broker and nothing to the system.

In [ ]:
import os
import pathlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from opentsx.client import RetrievalClient

RETRIEVAL = os.environ.get("OPENTSX_RETRIEVAL", "http://opentsx-retrieval.opentsx.svc:8080")
SYSTEM = "h2-chain"
client = RetrievalClient(RETRIEVAL)

# The file and how it is written
FILE = pathlib.Path("beispiel-windmessung.csv")
SEP, DECIMAL, COMMENT = ";", ",", "#"
TIME_COLUMNS = ["date", "time"]              # one column, or two that together give the time
TIME_FORMAT = "%d.%m.%Y %H:%M"
ZONE = "Europe/Berlin"                       # time zone of the file when it names none; "UTC" when it is in UTC
VALUE = "wind_kmh"
FILE_UNIT = "km/h"
TIME_MEANS = "start"                         # the time names the start of the interval ("start") or its end ("end")

# The signal the file is checked against
SIGNAL = "park/DWD-01759/wind_speed"
print("File:", FILE.resolve(), "| signal:", SIGNAL)

## 1. Read the file

Local time is converted to UTC. In the night of the clock change in autumn, the hour from 02:00 to 03:00 occurs
twice; it is assigned by the order of the rows. In spring, the hour from 02:00 to 03:00 does not exist.

A file with the German header `Datum;Uhrzeit;Wind_kmh` is read with the same settings: each name in `TIME_COLUMNS` and `VALUE` is looked up as written, then under its German name.

In [ ]:
GERMAN_HEADER = {"date": "Datum", "time": "Uhrzeit", "wind_kmh": "Wind_kmh"}
df = pd.read_csv(FILE, sep=SEP, decimal=DECIMAL, comment=COMMENT, dtype=str)


def column(setting, name):
    for found in (name, GERMAN_HEADER.get(name)):
        if found in df.columns:
            return found
    raise ValueError(f'{setting}: the file has no column "{name}"; expected one of {", ".join(df.columns)}')


stamp = df[[column("TIME_COLUMNS", c) for c in TIME_COLUMNS]].astype(str).agg(" ".join, axis=1)
local = pd.to_datetime(stamp, format=TIME_FORMAT, errors="coerce")
unreadable = int(local.isna().sum())
if ZONE == "UTC":
    ts = local.dt.tz_localize("UTC")
else:
    ts = local.dt.tz_localize(ZONE, ambiguous="infer", nonexistent="NaT").dt.tz_convert("UTC")
text = df[column("VALUE", VALUE)].astype(str).str.strip()
values = pd.to_numeric(text.str.replace(",", ".", regex=False) if DECIMAL == "," else text, errors="coerce")
data = pd.Series(values.values, index=ts).dropna()
data = data[data.index.notna()]
if data.empty:
    raise ValueError("FILE: no readable values; check SEP, DECIMAL, TIME_COLUMNS, TIME_FORMAT and VALUE")
print(f"{len(df)} rows read, {unreadable} times unreadable, {int(values.isna().sum())} values unreadable")
print(f"from {data.index.min()} to {data.index.max()} (UTC)")
data.head()

## 2. What the system expects

In [ ]:
sig = client.signals(SYSTEM).set_index("id").loc[SIGNAL]
print(f"{SIGNAL}: {sig['meaning']}")
print(f"Unit {sig['unit']}, plausible {sig['min']} to {sig['max']}, operating {sig['operating_min']} to {sig['operating_max']}, "
      f"interval {sig['interval']}, origin in the system {sig['origin']}")

## 3. The checks

Every check has a verdict, `ok`, `warning` or `error`, and a reason. A warning does not prevent the binding, but it
must be stated in the binding.

In [ ]:
FACTOR = {("km/h", "m/s"): 1 / 3.6, ("m/s", "m/s"): 1.0, ("kn", "m/s"): 0.514444, ("MW", "kW"): 1000.0, ("kW", "kW"): 1.0,
          ("°C", "Cel"): 1.0, ("Cel", "Cel"): 1.0, ("hPa", "hPa"): 1.0, ("EUR/MWh", "EUR/MWh"): 1.0, ("ct/kWh", "EUR/MWh"): 10.0}
checks = []


def check(name, verdict, reason):
    checks.append({"check": name, "verdict": verdict, "reason": reason})


# Unit
factor = FACTOR.get((FILE_UNIT, sig["unit"]))
if factor is None:
    check("Unit", "error", f"no conversion from {FILE_UNIT} to {sig['unit']} is known")
    converted = data
else:
    converted = data * factor
    check("Unit", "ok" if factor == 1 else "warning", f"{FILE_UNIT} to {sig['unit']}, factor {factor:.6g}")

# Interval
step = data.index.to_series().diff().dropna().mode().iloc[0]
expected = pd.Timedelta(sig["interval"]) if isinstance(sig["interval"], str) else None
if expected is None:
    check("Interval", "warning", f"the file has {step}; the signal names no interval")
elif step == expected:
    check("Interval", "ok", f"{step}")
elif step > expected:
    check("Interval", "warning", f"the file has {step}, the signal expects {expected}: coarser; a value does not become finer when it is repeated")
else:
    check("Interval", "warning", f"the file has {step}, the signal expects {expected}: finer; it is averaged when it is cleaned")

# Time: duplicate times and the meaning of the time
dupes = int(data.index.duplicated().sum())
check("Duplicate times", "ok" if dupes == 0 else "error", f"{dupes} duplicates")
check("Time zone", "ok" if ZONE == "UTC" else "warning", f"local time {ZONE}, converted to UTC" if ZONE != "UTC" else "UTC")
if TIME_MEANS == "end":
    converted.index = converted.index - step
    check("Time", "warning", "the file stamps the end of an interval; shifted to the start")
else:
    check("Time", "ok", "start of the interval, as in the system")

# Value range
outside = converted[(converted < sig["min"]) | (converted > sig["max"])] if pd.notna(sig["min"]) else converted.iloc[0:0]
check("Plausible range", "ok" if outside.empty else "warning",
      f"{len(outside)} values outside {sig['min']} to {sig['max']}" + (f", for example {outside.iloc[0]:.1f} at {outside.index[0]}" if len(outside) else ""))
inside = converted.drop(outside.index)
beyond = inside[(inside < sig["operating_min"]) | (inside > sig["operating_max"])] if pd.notna(sig["operating_min"]) else inside.iloc[0:0]
check("Operating range", "ok" if beyond.empty else "warning", f"{len(beyond)} plausible values outside the operating range {sig['operating_min']} to {sig['operating_max']}")

# Gaps at the interval of the file
full = pd.date_range(data.index.min(), data.index.max(), freq=step)
missing = full.difference(data.index)
check("Gaps", "ok" if len(missing) == 0 else "warning", f"{len(missing)} missing times of {len(full)} ({100 * len(missing) / len(full):.1f} %)")

verdicts = pd.DataFrame(checks)
verdicts

## 4. Next to the series of the system

The file, converted and without the values outside the plausible range, next to the clean series of the signal (the
raw one when there is no clean one) in the same period, both averaged to the interval of the file. If they match,
the file is what it claims to be; if they differ a lot, it measures something else or at another place.

In [ ]:
start, end = inside.index.min(), inside.index.max() + step
ids = sig["clean_series"] if isinstance(sig["clean_series"], str) else sig["raw_series"]
system = client.frame(series=ids.split(","), start=start, end=end).set_index("ts")["value"].sort_index()
pair = pd.concat({"File": inside.resample(step).mean(), "System": system.resample(step).mean()}, axis=1).dropna()
if len(pair) > 2:
    corr = pair["File"].corr(pair["System"])
    bias = (pair["File"] - pair["System"]).mean()
    check("Comparison with the system", "ok" if corr > 0.9 and abs(bias) < 0.1 * pair["System"].abs().mean() else "warning",
          f"correlation {corr:.3f}, mean difference {bias:+.2f} {sig['unit']} over {len(pair)} common times")
    fig, (a, b) = plt.subplots(1, 2, figsize=(14, 3.6), gridspec_kw={"width_ratios": [3, 1]})
    pair.plot(ax=a, lw=0.8)
    a.set_ylabel(sig["unit"])
    a.set_title("File and system")
    b.scatter(pair["System"], pair["File"], s=5)
    lim = [pair.min().min(), pair.max().max()]
    b.plot(lim, lim, color="grey", lw=0.8)
    b.set_xlabel("System")
    b.set_ylabel("File")
    plt.tight_layout()
    plt.show()
else:
    check("Comparison with the system", "warning", "no common times with the series of the system")
pd.DataFrame(checks).tail(1)

## 5. The verdict and the binding

An error prevents the binding. Warnings belong in the binding, so that everyone sees what the series is and what it
is not. The prepared copy is in UTC, in the unit of the signal and without the values outside the plausible range;
it lies in the directory of this notebook. The commands below bring it into the archive and into the topic of the
domain. Someone who is allowed to do that runs them, not this notebook.

In [ ]:
verdicts = pd.DataFrame(checks)
errors = verdicts[verdicts["verdict"] == "error"]
warnings = verdicts[verdicts["verdict"] == "warning"]
print("Verdict:", "not bindable" if len(errors) else ("bindable, with warnings" if len(warnings) else "bindable"))
for _, r in pd.concat([errors, warnings]).iterrows():
    print(f"  {r['verdict']}: {r['check']}: {r['reason']}")

OUT = pathlib.Path(f"checked-{SIGNAL.replace('/', '_')}.csv")
pd.DataFrame({"start": inside.index.strftime("%Y-%m-%dT%H:%M:%SZ"), "value": inside.round(4).values}).to_csv(OUT, index=False)
print(f"\nprepared: {OUT.resolve()} ({len(inside)} values)")

name = SIGNAL.replace("/", "-").lower()
metric = "partner." + SIGNAL.replace("/", ".").lower()
def iso(d):
    s = int(pd.Timedelta(d).total_seconds())
    return "PT" + (f"{s // 3600}H" if s % 3600 == 0 else f"{s // 60}M" if s % 60 == 0 else f"{s}S")


print(f"""
Binding, when the verdict allows it:

  bin/opentsx market import --file {OUT.name} --name {name} --metric {metric} --unit {sig['unit']} \\
      --origin MEASURED --resolution {iso(step)} --archive <archive>
  bin/opentsx market publish --archive <archive> --series partner/{name} --domain wind --kafscale <broker> --dry-run

and in the system document (or in the working copy on the place "Signals"):

  raw:
    source:
      kind: topic
      topic: tsx.wind.episodes.v2
      origin: MEASURED
      select: {{ metric: {metric} }}
    note: From a file of the operator, checked on {pd.Timestamp.now(tz='UTC'):%Y-%m-%d}. {'; '.join(warnings['reason'])}
""")